# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print title and description from the metadata
metadata = dataset.metadata
print("{}: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets and their fields and IDs as defined by Croissant's `@id`.

In [ ]:
# List record sets by @id
print('Available Record Sets:')
record_sets = []
for record_set in dataset.record_sets:
    print(f"- Name: {record_set.name}, @id: {record_set.id}")
    record_sets.append(record_set.id)
    # Show fields in the record set
    print('    Fields:')
    for field in record_set.fields:
        print(f"      - {field.name} (@id: {field.id}) | dataType: {field.data_type}")
if not record_sets:
    print('No RecordSets defined via Croissant. Attempting to load data via dataset.records()...')
    # Try fallback: enumerate generic records. The user must manually specify record set if more detailed schema is available.

## 3. Data Extraction
Load data from a specific record set (using its `@id`) into a DataFrame for analysis. The dataset record set structure dictates which fields are available for analysis.

In [ ]:
# For this dataset, if no record_sets in schema, use default (first/only) records
dataframes = {}
if record_sets:
    # There is at least one record set; use the first one
    record_set_id = record_sets[0]
    print(f"Extracting records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
else:
    # No record set; fallback to default
    record_set_id = None
    print("Extracting records using dataset.records() with default record set...")
    records = list(dataset.records())
    df = pd.DataFrame(records)
    dataframes['default'] = df

# Show dataframe columns
print(f"DataFrame columns:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data for simple statistics.

In [ ]:
import numpy as np

# Heuristically pick numeric field for demonstration by searching columns containing 'Abundance', 'MW', or similar numeric names
candidate_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64, float, int]]
if not candidate_numeric_fields:
    # Often Croissant data may be object dtype if not parsed, so try to coerce one
    for col in df.columns:
        try:
            converted = pd.to_numeric(df[col], errors='coerce')
            if converted.notnull().sum() > 0:
                candidate_numeric_fields.append(col)
        except Exception:
            continue
numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.columns[0]

print(f"Using numeric field: {numeric_field}")
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df[numeric_field].quantile(0.75) if df[numeric_field].notnull().sum() > 10 else 10

# Filtering
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (showing up to 5 rows):")
print(filtered_df.head())

# Normalizing the field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records (showing up to 5 rows):")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt to group by a likely categorical field
group_candidates = [col for col in df.columns if col.lower() in ['accession','gene','protein','sample','group','description']]
group_field = group_candidates[0] if group_candidates else None
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped data by {group_field} (mean of {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of numeric field
plt.figure(figsize=(7,4))
df[numeric_field].hist(bins=30)
plt.title(f'Histogram of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If group_field exists, plot mean by group
if group_field and group_field in filtered_df.columns:
    mean_by_group = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
    mean_by_group.plot(kind='bar', figsize=(9,4), title=f'Mean {numeric_field} by {group_field} (top 10)')
    plt.ylabel(f'Mean {numeric_field}')
    plt.xlabel(group_field)
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² mass spectrometry analysis dataset using `mlcroissant`, inspected its schema structure via record sets and fields, extracted tabular data, and performed a basic exploratory data analysis including filtering, normalization, grouping, and visualization.

- This data can be further analyzed for trends in protein abundance, comparative studies between groups, and detection of outliers or modification effects as suggested by the dataset's scientific context.
- For reproducible research and FAIR data practice, always cite the dataset by its identifier: **10.71728/senscience.vq4a-28xa** and refer to field/record set `@id` values in downstream analyses.
